# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**     Juan Pablo Moran|   |
| **Fecha**      |   |
| **Expediente**753760 |   |

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv), utiliza train-test-split de 70/30 y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.



In [1]:
import pandas as pd

df = pd.read_csv("Advertising.csv")
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [11]:
from sklearn.model_selection import train_test_split

X = df[["TV", "radio", "newspaper"]]
Y = df["sales"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.5, random_state=15)

### 1. Sin penalización

In [3]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, Y_train)

print(lr.intercept_)
print(lr.coef_)
print(lr.score(X_test, Y_test))

2.880255286331325
[0.04391531 0.20027962 0.00184368]
0.8649018906637792


### 2. Con penalización L2 (Ridge)

In [13]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=100)
ridge.fit(X_train, Y_train)

print(ridge.intercept_)
print(ridge.coef_)
print(ridge.score(X_test, Y_test))

2.5524023607535025
[0.04691193 0.18957631 0.0011522 ]
0.9105027712013366


### 3. Con penalización L1 (Lasso)

In [12]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1000)
lasso.fit(X_train, Y_train)

print(lasso.intercept_)
print(lasso.coef_)

13.889999999999997
[0. 0. 0.]


### Comparación variando alpha

In [6]:
alphas = [0.01, 0.1, 1, 10, 100]

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, Y_train)
    print("alpha =", a)
    print("coef:", ridge.coef_)
    print("R2:", ridge.score(X_test, Y_test))
    print()

alpha = 0.01
coef: [0.04391531 0.20027955 0.0018437 ]
R2: 0.8649019264857642

alpha = 0.1
coef: [0.04391531 0.20027894 0.00184382]
R2: 0.8649022488804243

alpha = 1
coef: [0.04391529 0.20027289 0.00184511]
R2: 0.8649054725096261

alpha = 10
coef: [0.04391504 0.2002124  0.00185798]
R2: 0.864937677079787

alpha = 100
coef: [0.04391256 0.19960953 0.00198596]
R2: 0.8652565685993675



In [7]:
for a in alphas:
    lasso = Lasso(alpha=a)
    lasso.fit(X_train, Y_train)
    print("alpha =", a)
    print("coef:", lasso.coef_)
    print("R2:", lasso.score(X_test, Y_test))
    print()

alpha = 0.01
coef: [0.0439147  0.20024296 0.00182846]
R2: 0.8649342220621365

alpha = 0.1
coef: [0.04390925 0.1999106  0.00169197]
R2: 0.865224901284483

alpha = 1
coef: [0.04385471 0.19658799 0.00032686]
R2: 0.8679417802015109

alpha = 10
coef: [0.04293969 0.15818074 0.        ]
R2: 0.8777165459820411

alpha = 100
coef: [0.03160993 0.         0.        ]
R2: 0.620910170381005



- Conforme aumenta alpha, los coeficientes se van encogiendo hacia cero en los dos modelos, pero no de la misma forma. En Ridge (L2) los coeficientes se reducen de manera suave y gradual, sin llegar nunca exactamente a cero (aunque alpha sea grande, newspaper sigue con un coeficiente chiquito pero distinto de cero). En Lasso (L1) el encogimiento es distinto: a partir de cierto valor de alpha, los coeficientes de las variables menos importantes se vuelven exactamente cero (por ejemplo newspaper se hace 0 desde alpha chico, y radio se hace 0 con alpha=10), es decir Lasso hace una especie de seleccion automatica de variables y se queda solo con las mas relevantes (TV en este caso).
- El R2 en test se mantiene practicamente igual entre el modelo sin penalizacion y Ridge/Lasso con alpha chico (0.01, 0.1, 1), porque casi no se esta penalizando nada todavia. Con alpha muy grande (100) el R2 de Lasso cae fuertemente (de ~0.86 a ~0.62) porque el modelo se vuelve demasiado simple (se queda solo con TV, o incluso menos) y ya no logra explicar bien las ventas: se sacrifica demasiada varianza a cambio de sesgo. Ridge en cambio es mas estable con alpha grande porque nunca elimina variables por completo, solo las encoge.
- En resumen: penalizar ayuda a controlar que un coeficiente se dispare de mas (como se menciono en la explicacion del principio), pero si alpha se pasa de grande, el modelo deja de ajustarse bien a los datos y el R2 baja. Hay que encontrar un punto intermedio de alpha.